In [2]:
import pandas as pd

In [3]:
input_tsv = 'pre_badge.tsv' # change this variable or rename .tsv file
df_in = pd.read_csv(input_tsv, sep='\t') # takes .tsv file generated by script in backend management commands
df_out = pd.DataFrame(columns=['first_name', 'last_name', 'affiliation', 'days', 'sunday', 'wed'])

In [4]:
def simp_days(days):
    days = days.replace(' ', '').replace('Th', 'R')
    if len(days) == 1:
        return days[0]
    if len(days) == 2:
        return days[0] + ' ' + days[1]
    day_to_num = {'M': 1, 'T': 2, 'W': 3, 'R': 4, 'F': 5}
    flag = True
    for i in range(len(days) - 1):
        if day_to_num[days[i]] != day_to_num[days[i+1]] - 1:
            flag = False
            break
    if flag:
        return days[0] + '-' + days[-1]
    else:
        return ' '.join(days)

In [5]:
rename_dic = {  'Massachusetts Institute of Technology': 'MIT',
                'University of Southern California (USC)': 'USC',
                'University of Southern California': 'USC',
                'Technion - Israel Institute of Technology': 'Technion - IIT',
                'State University of New York at Albany': 'SUNY Albany',
                'University of Minnesota Twin cities': 'UMN - Twin Cities',
                'University of California San Diego': 'UC San Diego',
                'Florida International University': 'FIU',
                'Delft University of Technology': 'TU Delft',
                'Toyota Technological Institute at Chicago': 'TTIC',
                'Georgia Institute of Technology': 'GaTech',
                'University of Wisconsin-Madison': 'UW-Madison',
                'Pohang University of Science and Technology (POSTECH)': 'POSTECH',
                'University of California Irvine': 'UC Irvine',
                'Indian Institute of Technology Madras': 'IIT Madras',
                'National University of Singapore': 'NUS',
                'University of Califronia, San Diego': 'UC San Diego',
                'University of Michigan, Ann Arbor': 'UMich',
                'University of Illinois at Urbana Champaign': 'UIUC',
                'University of Colorado at Colorado Springs': 'UCCS',
                'University of Pennsylvania': 'UPenn',
                'Japan Advanced Institute of Science and Technology': 'JAIST',
                'University of British Columbia': 'UBC',
                'Australian National University': 'ANU',
                'UC Los Angeles (UCLA)': 'UC Los Angeles',
                'Washington University in St. Louis': 'WashU',
                'Chinese University of Hong Kong': 'CUHK',
                'University of Illinois at Chicago': 'UIC',
                'University of Michigan- Ann Arbor': 'UMich',
                'Arizona State University': 'ASU',
                'San Diego State University': 'SDSU',
                'University of California, San Diego': 'UC San Diego',
                'University of California, Berkeley': 'UC Berkeley',
                'University of Souhern California': 'USC',
                'Washington State University': 'WSU',
                'California institute of technology': 'CalTech',
                'Broad Institute of MIT and Harvard': 'Broad Institute',
                'Center for Communications Research': 'CCRL',
                'Qualcomm Technologies Inc.': 'Qualcomm',
                'Scripps Institute of Oceanography': 'SIO',
                'Ilmenau University of Technology': 'TU Ilmenau',
                'University of New South Wales': 'UNSW',
                'The Ohio State University': 'Ohio State',
                'California State University, Sacramento': 'CSUS',
                'Missouri University of Science and Technology': 'Missouri S&T',
                
              }

rename_dic = {key.lower(): rename_dic[key] for key in rename_dic}

In [11]:
for index, row in df_in.iterrows():
    if pd.isna(row['first_name']) or pd.isna(row['last_name']):
        print('Participants with No Name:')
        print(row.to_dict())
        continue
    first_name = row['first_name']
    last_name = row['last_name']
    affliation = row['affiliation_title']
    if str(affliation).lower() in rename_dic:
        affliation = rename_dic[affliation.lower()]
    if pd.isna(row['badge_str']):
        badge_lst = []
    else:
        badge_lst = [s.strip() for s in str(row['badge_str']).split('|')]
        if len(badge_lst) == 1:
            days = simp_days(badge_lst[0])
            sunday = ''
            wed = ''
        elif len(badge_lst) == 2:
            if badge_lst[0] == '1' or badge_lst[0] == '2':
                days = simp_days(badge_lst[1])
                sunday = badge_lst[0]
                wed = ''
            else:
                days = simp_days(badge_lst[0])
                sunday = ''
                wed = badge_lst[1]
        elif len(badge_lst) == 3:
            sunday = badge_lst[0]
            days = simp_days(badge_lst[1])
            wed = badge_lst[2]
        wed = wed.replace('🥦', 'V').replace('🐟', 'F').replace('🐔', 'C')
    df_out = pd.concat([df_out, pd.DataFrame({'first_name': [first_name], 
                                              'last_name': [last_name], 
                                              'affiliation': [affliation], 
                                              'days': [days], 
                                              'sunday': [sunday], 
                                              'wed': [wed], 
                                              'sun_wed': [str(sunday) + wed]})], ignore_index=True)
df_out = df_out.sort_values(by='last_name')

Participants with No Name:
{'registrant_number': 131, 'first_name': nan, 'last_name': nan, 'affiliation_title': nan, 'attending_dates': 'T W Th ', 'sunday_reception_guests': 0, 'wednesday_banquet_guests': 1, 'saturday_italt_guests': 0, 'paid': False, 'fee_type_override': nan, 'badge_str': 'T W Th | 🐔'}


In [7]:
def filter_by_day(df, day):
    def helper_filter(s):
        day_to_num = {'M': 1, 'T': 2, 'W': 3, 'R': 4, 'F': 5}
        if '-' not in s:
            return day in s
        else:
            first_day = day_to_num[s[0]]
            last_day = day_to_num[s[-1]]
            target_day = day_to_num[day]
            return first_day < target_day and target_day < last_day
    return df.loc[df['days'].apply(helper_filter)]

In [8]:
print('Duplicate Names:')
for i in range(1, df_out.shape[0]):
    row1 = df_out.iloc[i]
    row2 = df_out.iloc[i - 1]
    if row1['first_name'].strip().lower() == row2['first_name'].strip().lower() and \
        row1['last_name'].strip().lower() == row2['last_name'].strip().lower():
        print(df_out['first_name'].iloc[i], df_out['last_name'].iloc[i])

Duplicate Names:
Fariba Abbasi
Ziv Aharoni
Pengfei Li
Mahdi Morafah
Mohamed Nafea
Jong-Seon No
Ian Roberts
Parastoo Sadeghi
Donya Saless
Ness Shroff


In [ ]:
print('Participants Missing Affiliation:')
for i in range(1, df_out.shape[0]):
    row = df_out.iloc[i]
    if pd.isna(row['affiliation']):
        print(df_out['first_name'].iloc[i], df_out['last_name'].iloc[i])

Missing Affiliation:


In [10]:
df_out.to_csv('post_badge.csv', index=False)